In [ ]:
#loading slides from 02_Weight_Optimization

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import tensorflow as tf

EXP2_DIR = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_02_weight_optimization"
)

selected_df = pd.read_csv(
    f"{EXP2_DIR}/best_selected_slices.csv"
)

print("Selected slices:", len(selected_df))
print("Subjects:", selected_df["SUB_ID"].nunique())
print(selected_df.columns.tolist())

Mounted at /content/drive
Selected slices: 30
Subjects: 6
['SUB_ID', 'SLICE_PATH', 'LABEL', 'SHAP_SCORE', 'PERMUTATION_SCORE', 'SHAP_NORM', 'PERMUTATION_NORM', 'COMBINED_SCORE', 'SLICE_RANK', 'BEST_SCORE', 'BEST_RANK']


In [ ]:
MODEL_PATH = (
    "/content/drive/MyDrive/ASD_XAI_Project/models/"
    "efficientnet_baseline.keras"
)

model = tf.keras.models.load_model(MODEL_PATH)

IMG_SIZE = 224

def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    return image

print("Model loaded!")
print(model.input_shape)

Model loaded!
(None, 224, 224, 3)


In [ ]:
from sklearn.metrics import accuracy_score

weighted_results = []

for sub_id, group in selected_df.groupby("SUB_ID"):

    images = np.stack([
        load_image(path).numpy()
        for path in group["SLICE_PATH"]
    ])

    predictions = model.predict(
        images,
        batch_size=16,
        verbose=0
    ).reshape(-1)

    weights = group["BEST_SCORE"].to_numpy()

    weighted_probability = np.sum(
        predictions * weights
    ) / np.sum(weights)

    predicted_label = (
        "ASD"
        if weighted_probability >= 0.5
        else "Control"
    )

    true_label = group["LABEL"].iloc[0]

    weighted_results.append({
        "SUB_ID": sub_id,
        "TRUE_LABEL": true_label,
        "PREDICTED_LABEL": predicted_label,
        "WEIGHTED_PROBABILITY": weighted_probability,
        "NUM_SLICES": len(group)
    })


weighted_results_df = pd.DataFrame(weighted_results)

weighted_accuracy = accuracy_score(
    weighted_results_df["TRUE_LABEL"],
    weighted_results_df["PREDICTED_LABEL"]
)

print("=== WEIGHTED AGGREGATION RESULTS ===")
print(weighted_results_df)

print("\nWeighted subject-level accuracy:")
print(weighted_accuracy)

=== WEIGHTED AGGREGATION RESULTS ===
   SUB_ID TRUE_LABEL PREDICTED_LABEL  WEIGHTED_PROBABILITY  NUM_SLICES
0   51460        ASD         Control              0.302191           5
1   51463        ASD             ASD              0.630855           5
2   51466        ASD             ASD              0.561437           5
3   51475    Control             ASD              0.724836           5
4   51476    Control         Control              0.446400           5
5   51487    Control             ASD              0.583791           5

Weighted subject-level accuracy:
0.5
